# 06 - Data Leakage: Assignment Completion

This notebook completes the data leakage assignment with practical examples and reproducible checks.

## Learning Goals
- Define data leakage precisely
- Identify target, temporal, and contamination leakage
- Run side-by-side experiments showing honest vs leaked evaluation
- Apply a prevention checklist before training any model

## Core Rule

A model must only learn from information available at the exact prediction moment in the real world.

If a feature depends on future events, post-outcome signals, or test-set knowledge, evaluation is invalid.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

np.random.seed(42)

def evaluate_binary_model(X, y, test_size=0.2, random_state=42):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )

    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    return {
        "accuracy": accuracy_score(y_test, preds),
        "f1": f1_score(y_test, preds),
    }

## Example 1 - Target Leakage

Scenario: Predict customer churn.

Leaky feature: `cancellation_reason_present` (known only after churn).

In [ ]:
n = 2000
tenure = np.random.randint(1, 72, n)
monthly_charges = np.random.normal(70, 20, n).clip(15, 200)
support_calls = np.random.poisson(2.5, n)

# True churn process (no leakage)
logit = -2.5 + 0.02 * monthly_charges + 0.35 * support_calls - 0.02 * tenure
prob = 1 / (1 + np.exp(-logit))
churn = (np.random.rand(n) < prob).astype(int)

# Leaky feature derived from target outcome (post-event artifact)
cancellation_reason_present = churn.copy()

df_churn = pd.DataFrame({
    "tenure": tenure,
    "monthly_charges": monthly_charges,
    "support_calls_last_90_days": support_calls,
    "cancellation_reason_present": cancellation_reason_present,
    "churn": churn,
})

X_honest = df_churn[["tenure", "monthly_charges", "support_calls_last_90_days"]]
X_leaky = df_churn[["tenure", "monthly_charges", "support_calls_last_90_days", "cancellation_reason_present"]]
y = df_churn["churn"]

honest_scores = evaluate_binary_model(X_honest, y)
leaky_scores = evaluate_binary_model(X_leaky, y)

pd.DataFrame([
    {"setup": "Honest features", **honest_scores},
    {"setup": "Leaky features", **leaky_scores},
])

Expected pattern: the leaky setup appears dramatically better because it encodes the answer.

Rule: if a feature only exists after the target occurs, remove it.

## Example 2 - Temporal Leakage

Scenario: Predict readmission at discharge time.

Leaky feature: `follow_up_booked` if it is scheduled after discharge decisions and risk workflows.

In [ ]:
n = 2500
age = np.random.randint(18, 90, n)
length_of_stay = np.random.randint(1, 20, n)
severity = np.random.randint(1, 6, n)

logit = -3 + 0.025 * age + 0.10 * length_of_stay + 0.55 * severity
prob = 1 / (1 + np.exp(-logit))
readmitted = (np.random.rand(n) < prob).astype(int)

# Future-informed downstream artifact (simulated)
follow_up_booked = ((readmitted == 1) | (severity >= 4)).astype(int)

df_hospital = pd.DataFrame({
    "age": age,
    "length_of_stay": length_of_stay,
    "severity": severity,
    "follow_up_booked": follow_up_booked,
    "readmitted": readmitted,
})

X_honest = df_hospital[["age", "length_of_stay", "severity"]]
X_temporal_leak = df_hospital[["age", "length_of_stay", "severity", "follow_up_booked"]]
y = df_hospital["readmitted"]

honest_scores = evaluate_binary_model(X_honest, y)
leaky_scores = evaluate_binary_model(X_temporal_leak, y)

pd.DataFrame([
    {"setup": "Prediction-time valid", **honest_scores},
    {"setup": "Temporal leakage", **leaky_scores},
])

Rule: if it occurs after the prediction decision point, it is not a valid feature.

## Example 3 - Train/Test Contamination

Incorrect workflow: fit preprocessing on the full dataset before splitting.

Correct workflow: split first, then fit preprocessors using training data only.

In [ ]:
# Synthetic data with scale-sensitive model behavior
n = 3000
x1 = np.random.normal(0, 1, n)
x2 = np.random.normal(0, 100, n)
x3 = np.random.normal(5, 2, n)
y = ((0.8 * x1 + 0.01 * x2 - 0.3 * x3 + np.random.normal(0, 0.8, n)) > 0).astype(int)

X = pd.DataFrame({"x1": x1, "x2": x2, "x3": x3})

# Wrong: scale before split
scaler_wrong = StandardScaler()
X_scaled_wrong = scaler_wrong.fit_transform(X)
X_train_w, X_test_w, y_train_w, y_test_w = train_test_split(
    X_scaled_wrong, y, test_size=0.2, random_state=42, stratify=y
)
model_wrong = LogisticRegression(max_iter=1000).fit(X_train_w, y_train_w)
pred_w = model_wrong.predict(X_test_w)

# Correct: split first, then scale train only
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
scaler_correct = StandardScaler()
X_train_c_scaled = scaler_correct.fit_transform(X_train_c)
X_test_c_scaled = scaler_correct.transform(X_test_c)
model_correct = LogisticRegression(max_iter=1000).fit(X_train_c_scaled, y_train_c)
pred_c = model_correct.predict(X_test_c_scaled)

pd.DataFrame([
    {"workflow": "Wrong: fit scaler before split", "accuracy": accuracy_score(y_test_w, pred_w), "f1": f1_score(y_test_w, pred_w)},
    {"workflow": "Correct: split then fit scaler on train", "accuracy": accuracy_score(y_test_c, pred_c), "f1": f1_score(y_test_c, pred_c)},
])

Even when score differences are small, the boundary rule still matters: all fitting operations must only see training data.

## Prediction Moment Test (Use Before Every Training Run)

Ask for each feature:
1. Does this exist at the exact prediction time?
2. Could this have been created using target or future events?
3. Could this contain information from held-out/test records?

If any answer is uncertain, quarantine the feature and investigate.

In [ ]:
def leakage_screen(feature_notes: dict):
    """Simple feature governance helper for assignment use."""
    decisions = []
    for feature, meta in feature_notes.items():
        valid = (
            meta.get("available_at_prediction_time", False)
            and not meta.get("target_derived", False)
            and not meta.get("uses_test_information", False)
        )
        decisions.append({
            "feature": feature,
            "keep": valid,
            "reason": "valid" if valid else "review/remove"
        })
    return pd.DataFrame(decisions).sort_values(by=["keep", "feature"], ascending=[False, True])

feature_notes = {
    "tenure": {"available_at_prediction_time": True, "target_derived": False, "uses_test_information": False},
    "monthly_charges": {"available_at_prediction_time": True, "target_derived": False, "uses_test_information": False},
    "cancellation_reason_present": {"available_at_prediction_time": False, "target_derived": True, "uses_test_information": False},
    "regional_churn_rate_global": {"available_at_prediction_time": True, "target_derived": False, "uses_test_information": True},
}

leakage_screen(feature_notes)

## Assignment Summary (Submission-Ready)

- Data leakage is a boundary violation, not a syntax error.
- Target leakage and temporal leakage can produce unrealistic metrics.
- Train/test contamination invalidates evaluation even when labels are untouched.
- Correct order is mandatory: split first, fit on train only, transform test with train-fitted objects.
- The prediction moment test is the most reliable practical defense.

## Bonus Content

- Kaggle Tutorial on Data Leakage: https://www.kaggle.com/code/alexisbcook/data-leakage
- Google Machine Learning Crash Course - Data Preparation: https://developers.google.com/machine-learning/crash-course/data-prep